<a href="https://colab.research.google.com/github/Qaiserfarooq285/FA25-AI/blob/main/ai_engineer_portfolio_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Engineer Portfolio Search

This notebook loads a dataset of AI Engineer portfolios and provides search/filter functions to find candidates by **skills**, **experience level**, **location**, **specialization**, and **salary range**.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Load the portfolio dataset
PATH = 'ai_engineer_portfolios.csv'
df = pd.read_csv(PATH)

# Convert Skills column from comma-separated string to list
df['SkillList'] = df['Skills'].apply(lambda s: [sk.strip() for sk in s.split(',')])

print(f"Total portfolios loaded: {len(df)}")
df.head()

## 2. Dataset Overview

In [ ]:
# Basic statistics
df[['YearsExperience', 'Salary']].describe()

In [ ]:
# Distribution of experience levels
print("Experience Level Distribution:")
print(df['ExperienceLevel'].value_counts())

print("\nSpecialization Distribution:")
print(df['Specialization'].value_counts())

print("\nLocation Distribution:")
print(df['Location'].value_counts())

## 3. Exploratory Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Salary distribution by experience level
sns.boxplot(data=df, x='ExperienceLevel', y='Salary',
            order=['Junior', 'Mid', 'Senior'], ax=axes[0],
            hue='ExperienceLevel', legend=False, palette='Set2')
axes[0].set_title('Salary Distribution by Experience Level')
axes[0].set_ylabel('Salary (USD)')
axes[0].set_xlabel('Experience Level')

# Specialization count
spec_counts = df['Specialization'].value_counts()
axes[1].bar(spec_counts.index, spec_counts.values, color='steelblue')
axes[1].set_title('Number of Engineers per Specialization')
axes[1].set_xlabel('Specialization')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 most common skills across all portfolios
all_skills = [skill for skills in df['SkillList'] for skill in skills]
skill_counts = Counter(all_skills)
top_skills = pd.DataFrame(skill_counts.most_common(15), columns=['Skill', 'Count'])

plt.figure(figsize=(10, 5))
sns.barplot(data=top_skills, x='Skill', y='Count',
            hue='Skill', legend=False, palette='viridis')
plt.title('Top 15 Most Common Skills in AI Engineer Portfolios')
plt.xlabel('Skill')
plt.ylabel('Number of Engineers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Search Functions

In [ ]:
def search_by_skills(df, required_skills, match_all=False):
    """
    Search portfolios by one or more skills.

    Parameters
    ----------
    df : pd.DataFrame
        Portfolio DataFrame (must contain a 'SkillList' column).
    required_skills : list of str
        Skills to search for (case-insensitive).
    match_all : bool
        If True, the engineer must have ALL listed skills.
        If False (default), any one matching skill is enough.

    Returns
    -------
    pd.DataFrame
        Filtered portfolio rows.
    """
    required_lower = [s.strip().lower() for s in required_skills]

    def matches(skill_list):
        engineer_skills = [s.lower() for s in skill_list]
        if match_all:
            return all(req in engineer_skills for req in required_lower)
        return any(req in engineer_skills for req in required_lower)

    result = df[df['SkillList'].apply(matches)].copy()
    return result.drop(columns=['SkillList'])


def filter_by_experience_level(df, level):
    """
    Filter portfolios by experience level: 'Junior', 'Mid', or 'Senior'.
    """
    return df[df['ExperienceLevel'].str.lower() == level.strip().lower()].drop(columns=['SkillList'])


def filter_by_location(df, location):
    """
    Filter portfolios by city/location (case-insensitive, partial match).
    """
    return df[df['Location'].str.lower().str.contains(location.strip().lower())].drop(columns=['SkillList'])


def filter_by_salary_range(df, min_salary=0, max_salary=float('inf')):
    """
    Filter portfolios by salary range (inclusive).
    """
    return df[(df['Salary'] >= min_salary) & (df['Salary'] <= max_salary)].drop(columns=['SkillList'])


def filter_by_specialization(df, specialization):
    """
    Filter portfolios by specialization (case-insensitive, partial match).
    """
    return df[df['Specialization'].str.lower().str.contains(specialization.strip().lower())].drop(columns=['SkillList'])


def search_portfolios(df, skills=None, match_all_skills=False,
                      experience_level=None, location=None,
                      min_salary=None, max_salary=None,
                      specialization=None):
    """
    Combined search across all filters.

    Parameters
    ----------
    df : pd.DataFrame
        Portfolio DataFrame.
    skills : list of str, optional
        Skills to filter by.
    match_all_skills : bool
        Whether ALL skills must match (True) or ANY (False).
    experience_level : str, optional
        'Junior', 'Mid', or 'Senior'.
    location : str, optional
        City or partial city name.
    min_salary : int, optional
        Minimum salary (inclusive).
    max_salary : int, optional
        Maximum salary (inclusive).
    specialization : str, optional
        Specialization keyword (e.g. 'NLP', 'Computer Vision').

    Returns
    -------
    pd.DataFrame
        Matching portfolio rows sorted by salary descending.
    """
    result = df.copy()

    if skills:
        required_lower = [s.strip().lower() for s in skills]
        def skill_match(skill_list):
            eng = [s.lower() for s in skill_list]
            if match_all_skills:
                return all(r in eng for r in required_lower)
            return any(r in eng for r in required_lower)
        result = result[result['SkillList'].apply(skill_match)]

    if experience_level:
        result = result[result['ExperienceLevel'].str.lower() == experience_level.strip().lower()]

    if location:
        result = result[result['Location'].str.lower().str.contains(location.strip().lower())]

    if min_salary is not None:
        result = result[result['Salary'] >= min_salary]

    if max_salary is not None:
        result = result[result['Salary'] <= max_salary]

    if specialization:
        result = result[result['Specialization'].str.lower().str.contains(specialization.strip().lower())]

    return result.drop(columns=['SkillList']).sort_values('Salary', ascending=False).reset_index(drop=True)


print("Search functions defined successfully.")

## 5. Example Searches

### 5a. Search by a single skill – PyTorch

In [ ]:
pytorch_engineers = search_by_skills(df, ['PyTorch'])
print(f"Engineers with PyTorch: {len(pytorch_engineers)}")
pytorch_engineers[['Name', 'Location', 'ExperienceLevel', 'Specialization', 'Skills', 'Salary']]

### 5b. Search by multiple skills (must have ALL of them)

In [ ]:
# Find engineers who know BOTH Python AND Docker AND Kubernetes
devops_ai = search_by_skills(df, ['Python', 'Docker', 'Kubernetes'], match_all=True)
print(f"Engineers with Python + Docker + Kubernetes: {len(devops_ai)}")
devops_ai[['Name', 'Location', 'ExperienceLevel', 'Specialization', 'Skills', 'Salary']]

### 5c. Filter by experience level – Senior only

In [ ]:
senior_engineers = filter_by_experience_level(df, 'Senior')
print(f"Senior AI Engineers: {len(senior_engineers)}")
senior_engineers[['Name', 'Location', 'Specialization', 'Skills', 'Salary']]

### 5d. Filter by location – New York

In [ ]:
ny_engineers = filter_by_location(df, 'New York')
print(f"AI Engineers in New York: {len(ny_engineers)}")
ny_engineers[['Name', 'ExperienceLevel', 'Specialization', 'Skills', 'Salary']]

### 5e. Filter by salary range – $120,000 to $160,000

In [ ]:
mid_salary = filter_by_salary_range(df, min_salary=120000, max_salary=160000)
print(f"Engineers earning $120k–$160k: {len(mid_salary)}")
mid_salary[['Name', 'Location', 'ExperienceLevel', 'Specialization', 'Skills', 'Salary']]

### 5f. Filter by specialization – NLP

In [ ]:
nlp_engineers = filter_by_specialization(df, 'NLP')
print(f"NLP-specialised Engineers: {len(nlp_engineers)}")
nlp_engineers[['Name', 'Location', 'ExperienceLevel', 'Skills', 'Salary']]

### 5g. Combined search – Senior NLP engineers in San Francisco earning > $140k

In [ ]:
results = search_portfolios(
    df,
    experience_level='Senior',
    specialization='NLP',
    location='San Francisco',
    min_salary=140000
)
print(f"Matching portfolios: {len(results)}")
results[['Name', 'Location', 'ExperienceLevel', 'Specialization', 'Skills', 'Salary']]

### 5h. Combined search – Mid-level engineers with PyTorch or TensorFlow

In [ ]:
results = search_portfolios(
    df,
    skills=['PyTorch', 'TensorFlow'],
    match_all_skills=False,
    experience_level='Mid'
)
print(f"Matching portfolios: {len(results)}")
results[['Name', 'Location', 'Specialization', 'Skills', 'Salary']]

## 6. Salary Analysis by Specialization

In [ ]:
salary_by_spec = df.groupby('Specialization')['Salary'].agg(['mean', 'min', 'max']).sort_values('mean', ascending=False)
salary_by_spec.columns = ['Avg Salary', 'Min Salary', 'Max Salary']
salary_by_spec = salary_by_spec.round(0).astype(int)
print("Salary Statistics by Specialization:")
salary_by_spec

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='Specialization', y='Salary',
            order=salary_by_spec.index, hue='Specialization', legend=False,
            palette='coolwarm', errorbar='sd')
plt.title('Average Salary by Specialization (with Std Dev)')
plt.xlabel('Specialization')
plt.ylabel('Salary (USD)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()